In [ ]:
# CELL 0 (new notebook only): place ckpt_test/ and prep_test/ in /kaggle/working,
# where Step 2 v3 reads them. Works whether Kaggle unpacked the uploaded zips or not.
import shutil, zipfile
from pathlib import Path

W, I = Path("/kaggle/working"), Path("/kaggle/input")

def place(name, marker):
    dest = W / name
    found = {}
    for f in sorted(I.rglob(marker)):                     # unpacked uploads
        if f.parent.parent.name == name:
            found.setdefault(f.parent.name, f.parent)
    for z in sorted(I.rglob("*.zip")):                   # uploads kept as zips
        with zipfile.ZipFile(z) as zf:
            members = [n for n in zf.namelist() if n.startswith(f"{name}/")]
            countries = {n.split("/")[1] for n in members if n.count("/") >= 2}
            if members and not countries <= set(found):
                zf.extractall(W)
                for c in countries:
                    found.setdefault(c, W / name / c)
    for c, src in sorted(found.items()):
        if not (dest / c).exists():
            shutil.copytree(src, dest / c)
    if not dest.exists():
        print(f"{name}: NOT FOUND — attach its dataset"); return
    for c in sorted(p.name for p in dest.iterdir() if p.is_dir()):
        files = sorted(f.name for f in (dest / c).iterdir())
        size = sum(f.stat().st_size for f in (dest / c).iterdir()) / 1e9
        print(f"{name}/{c:<7} {size:5.2f} GB   {', '.join(files)}")

place("ckpt_test", "qids.npy")
place("prep_test", "q.parquet")
used = sum(f.stat().st_size for f in W.rglob("*") if f.is_file()) / 1e9
print(f"/kaggle/working: {used:.1f} GB used of 20 GB")